In [2]:
import openai
import pandas as pd
import tiktoken


from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import FieldCondition, MatchAny, Filter
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [3]:
qdrant_client.create_collection(
    collection_name="Amazon-reviews-collection-01",
    vectors_config={
        "text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE)
    }
)

True

In [4]:
qdrant_client.create_payload_index(
    collection_name="Amazon-reviews-collection-01",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [4]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [5]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

In [6]:
df_reviews = pd.read_json("../../data/Electronics_with_category_sample_1000.jsonl", lines=True)

In [7]:
df_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,3,marginal sound quality,this really deserves 3 stars. its not terribl...,[],B0047T79VS,B0047T79VS,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,2012-08-08 06:08:03.000,0,True
1,5,Five Stars,Great hard drive for the price! Installed with...,[],B000Q85WOK,B000Q85WOK,AEFKF6R2GUSK2AWPSWRR4ZO36JVQ,2018-07-20 07:34:48.895,0,True
2,3,Too large for any speaker I have in my arsenal,I have 2 JBL and an LG portable Bluetooth spea...,[],B07ZWCTNJH,B07ZWCTNJH,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,2022-02-26 16:04:57.397,0,True
3,1,Junk that doesn’t work,I would give this negative 0 stars if I could....,[],B09KTMJVJW,B09KTMJVJW,AEI4YPQZG6TZOWHDFP7I22S4MQZQ,2021-12-20 21:37:47.719,1,True
4,5,works well,"works very well. Has a very small USB nub, wh...",[],B005HQ5138,B005HQ5138,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2014-09-20 18:03:16.000,0,True


In [8]:
len(df_reviews)

167138

In [10]:
def preprocess_reviews_data(row):
    return f"{row['title']} {row['text']}"

In [16]:
def count_tokens(row):
    encoding = tiktoken.encoding_for_model("text-embedding-3-small")
    return len(encoding.encode(row['preprocessed_data']))

In [14]:
count_tokens("How are you doing today?")

6

In [17]:
df_reviews['preprocessed_data'] =  df_reviews.apply(preprocess_reviews_data, axis=1)
df_reviews['token_count'] = df_reviews.apply(count_tokens, axis=1)

In [18]:
df_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,preprocessed_data,token_count
0,3,marginal sound quality,this really deserves 3 stars. its not terribl...,[],B0047T79VS,B0047T79VS,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,2012-08-08 06:08:03.000,0,True,marginal sound quality this really deserves 3 ...,63
1,5,Five Stars,Great hard drive for the price! Installed with...,[],B000Q85WOK,B000Q85WOK,AEFKF6R2GUSK2AWPSWRR4ZO36JVQ,2018-07-20 07:34:48.895,0,True,Five Stars Great hard drive for the price! Ins...,24
2,3,Too large for any speaker I have in my arsenal,I have 2 JBL and an LG portable Bluetooth spea...,[],B07ZWCTNJH,B07ZWCTNJH,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,2022-02-26 16:04:57.397,0,True,Too large for any speaker I have in my arsenal...,87
3,1,Junk that doesn’t work,I would give this negative 0 stars if I could....,[],B09KTMJVJW,B09KTMJVJW,AEI4YPQZG6TZOWHDFP7I22S4MQZQ,2021-12-20 21:37:47.719,1,True,Junk that doesn’t work I would give this negat...,36
4,5,works well,"works very well. Has a very small USB nub, wh...",[],B005HQ5138,B005HQ5138,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2014-09-20 18:03:16.000,0,True,works well works very well. Has a very small ...,66


In [19]:
df_reviews = df_reviews[df_reviews['token_count'] < 8192]

In [20]:
len(df_reviews)

167138

In [21]:
total_tokens = df_reviews['token_count'].sum()
total_tokens

np.int64(10018132)

In [23]:
df_reviews_to_embed = df_reviews[['preprocessed_data', 'parent_asin']]

In [24]:
df_reviews_to_embed.head()

,preprocessed_data,parent_asin
0,marginal sound quality this really deserves 3 ...,B0047T79VS
1,Five Stars Great hard drive for the price! Ins...,B000Q85WOK
2,Too large for any speaker I have in my arsenal...,B07ZWCTNJH
3,Junk that doesn’t work I would give this negat...,B09KTMJVJW
4,works well works very well. Has a very small ...,B005HQ5138


In [26]:
data_to_embed = df_reviews_to_embed.to_dict(orient="records")
data_to_embed

[{'preprocessed_data': 'marginal sound quality this really deserves 3 stars.  its not terrible, but its not great.  first is sound quality.  its not terrible.  2nd is limited distance.  its not terrible, but not great.<br /><br />I wish I had found a better product.',
  'parent_asin': 'B0047T79VS'},
 {'preprocessed_data': "Five Stars Great hard drive for the price! Installed with no issues in wife's Dell computer: She's very satisfied:",
  'parent_asin': 'B000Q85WOK'},
 {'preprocessed_data': "Too large for any speaker I have in my arsenal I have 2 JBL and an LG portable Bluetooth speaker and this was too large for any of them. One of the JBLs was barely long enough to be held by the straps but I didn't feel confident in how secure it was. The other 2 were just 2 short. I didn't think they were small speakers but apparently they are too small for this Mount",
  'parent_asin': 'B07ZWCTNJH'},
 {'preprocessed_data': 'Junk that doesn’t work I would give this negative 0 stars if I could. Doe

In [27]:
len(data_to_embed)

167138

### Embded

In [ ]:
text_to_embed_reviews = [item['preprocessed_data'] for item in data_to_embed]
text_to_embed_reviews

['marginal sound quality this really deserves 3 stars.  its not terrible, but its not great.  first is sound quality.  its not terrible.  2nd is limited distance.  its not terrible, but not great.<br /><br />I wish I had found a better product.',
 "Five Stars Great hard drive for the price! Installed with no issues in wife's Dell computer: She's very satisfied:",
 "Too large for any speaker I have in my arsenal I have 2 JBL and an LG portable Bluetooth speaker and this was too large for any of them. One of the JBLs was barely long enough to be held by the straps but I didn't feel confident in how secure it was. The other 2 were just 2 short. I didn't think they were small speakers but apparently they are too small for this Mount",
 'Junk that doesn’t work I would give this negative 0 stars if I could. Does not work at all. Shame on you for stealing $10 from unsuspecting people.',
 "works well works very well.  Has a very small USB nub, which you can store inside the mouse, or leave on 

In [29]:
embeddings = get_embeddings_batch(text_to_embed_reviews, batch_size=500)

Processed 500 of 167138
Processed 1000 of 167138
Processed 1500 of 167138
Processed 2000 of 167138
Processed 2500 of 167138
Processed 3000 of 167138
Processed 3500 of 167138
Processed 4000 of 167138
Processed 4500 of 167138
Processed 5000 of 167138
Processed 5500 of 167138
Processed 6000 of 167138
Processed 6500 of 167138
Processed 7000 of 167138
Processed 7500 of 167138
Processed 8000 of 167138
Processed 8500 of 167138
Processed 9000 of 167138
Processed 9500 of 167138
Processed 10000 of 167138
Processed 10500 of 167138
Processed 11000 of 167138
Processed 11500 of 167138
Processed 12000 of 167138
Processed 12500 of 167138
Processed 13000 of 167138
Processed 13500 of 167138
Processed 14000 of 167138
Processed 14500 of 167138
Processed 15000 of 167138
Processed 15500 of 167138
Processed 16000 of 167138
Processed 16500 of 167138
Processed 17000 of 167138
Processed 17500 of 167138
Processed 18000 of 167138
Processed 18500 of 167138
Processed 19000 of 167138
Processed 19500 of 167138
Proces

In [30]:
len(embeddings)

167138

In [32]:
from typing import Any


pointstructs = []
i=1
for embedding, data in zip(embeddings, data_to_embed):
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-3-small": embedding
            },
            payload=data
        )
    )
    i += 1

In [33]:
pointstructs[0].vector

{'text-embedding-3-small': [0.0029163360595703125,
  0.0262298583984375,
  -0.006984710693359375,
  -0.0176239013671875,
  -0.047576904296875,
  -0.042144775390625,
  0.036956787109375,
  -0.017059326171875,
  0.026397705078125,
  0.016845703125,
  -0.02752685546875,
  -0.03546142578125,
  -0.06414794921875,
  0.0679931640625,
  0.00897216796875,
  0.0244598388671875,
  -0.033416748046875,
  -0.038482666015625,
  -0.0024356842041015625,
  0.0247955322265625,
  -0.005519866943359375,
  0.056854248046875,
  0.0007410049438476562,
  -0.0034427642822265625,
  0.0160980224609375,
  -0.0170745849609375,
  0.033111572265625,
  0.0187530517578125,
  0.032440185546875,
  0.0025844573974609375,
  -0.016937255859375,
  -0.03582763671875,
  -0.0018634796142578125,
  -0.052764892578125,
  -0.006114959716796875,
  0.0090789794921875,
  -0.0027179718017578125,
  -0.02593994140625,
  0.0159759521484375,
  -0.042449951171875,
  -0.0106658935546875,
  0.07763671875,
  0.0017871856689453125,
  -0.0030441

In [34]:
batch_size_qdrant = 100
counter = 1
for i in range(0, len(pointstructs), batch_size_qdrant):
    batch = pointstructs[i:i + batch_size_qdrant]
    qdrant_client.upsert(
        collection_name="Amazon-reviews-collection-01",
        points=batch,
        wait=True
    )
    print(f"Processed {counter * batch_size_qdrant} of {len(pointstructs)}")
    counter += 1

Processed 100 of 167138
Processed 200 of 167138
Processed 300 of 167138
Processed 400 of 167138
Processed 500 of 167138
Processed 600 of 167138
Processed 700 of 167138
Processed 800 of 167138
Processed 900 of 167138
Processed 1000 of 167138
Processed 1100 of 167138
Processed 1200 of 167138
Processed 1300 of 167138
Processed 1400 of 167138
Processed 1500 of 167138
Processed 1600 of 167138
Processed 1700 of 167138
Processed 1800 of 167138
Processed 1900 of 167138
Processed 2000 of 167138
Processed 2100 of 167138
Processed 2200 of 167138
Processed 2300 of 167138
Processed 2400 of 167138
Processed 2500 of 167138
Processed 2600 of 167138
Processed 2700 of 167138
Processed 2800 of 167138
Processed 2900 of 167138
Processed 3000 of 167138
Processed 3100 of 167138
Processed 3200 of 167138
Processed 3300 of 167138
Processed 3400 of 167138
Processed 3500 of 167138
Processed 3600 of 167138
Processed 3700 of 167138
Processed 3800 of 167138
Processed 3900 of 167138
Processed 4000 of 167138
Processed

### A function to run search against reviews on a prefiltered set of product IDs

In [35]:
def retrieve_prefiltered_reviews_data(query, parent_asins, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-reviews-collection-01",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )
    
    return results

In [40]:
reviews = retrieve_prefiltered_reviews_data("bad_quality", ["B0047T79VS"])
reviews

QueryResponse(points=[ScoredPoint(id=3362, version=36, score=0.5, payload={'preprocessed_data': 'Very poor sound quality ! Very poor sound quality !', 'parent_asin': 'B0047T79VS'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=3963, version=42, score=0.33333334, payload={'preprocessed_data': 'the quality its not very good the quality its not bad but its good<br /><br />i try it with iphone and playbook (no any problem)<br /><br />with banasonic sterio (no any problem)', 'parent_asin': 'B0047T79VS'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=3286, version=35, score=0.25, payload={'preprocessed_data': "Very Low Quality Avoid this. Doesn't work. Good manufacturer, bad nonfunctional product. The sound quality is poor and this product's ability to use the Bluetooth protocol is limited.", 'parent_asin': 'B0047T79VS'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=4680, version=49, score=0.2, payload={'preprocessed_data': 'One Star Sound 

In [42]:
reviews.points

[ScoredPoint(id=3362, version=36, score=0.5, payload={'preprocessed_data': 'Very poor sound quality ! Very poor sound quality !', 'parent_asin': 'B0047T79VS'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3099, version=33, score=0.33333334, payload={'preprocessed_data': 'Low quality sound Low quality sound! This is the worse quality you can get, is staticky. I used it for my laptop to play in my home theater and it sucks because it take away sound, base, quality & volumen. Do not buy if you are looking to use it for music', 'parent_asin': 'B0047T79VS'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=4548, version=48, score=0.25, payload={'preprocessed_data': "Bad quality sound. Don't waste your money.  Sound quality is bad.  It degrades the already compressed mp3 and sounds like AM radio.", 'parent_asin': 'B0047T79VS'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3286, version=35, score=0.2, payload={'preprocessed_data': "Very Low

In [41]:
reviews = retrieve_prefiltered_reviews_data("bad quality", ["B0047T79VS"])
reviews

QueryResponse(points=[ScoredPoint(id=3362, version=36, score=0.5, payload={'preprocessed_data': 'Very poor sound quality ! Very poor sound quality !', 'parent_asin': 'B0047T79VS'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=3099, version=33, score=0.33333334, payload={'preprocessed_data': 'Low quality sound Low quality sound! This is the worse quality you can get, is staticky. I used it for my laptop to play in my home theater and it sucks because it take away sound, base, quality & volumen. Do not buy if you are looking to use it for music', 'parent_asin': 'B0047T79VS'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=4548, version=48, score=0.25, payload={'preprocessed_data': "Bad quality sound. Don't waste your money.  Sound quality is bad.  It degrades the already compressed mp3 and sounds like AM radio.", 'parent_asin': 'B0047T79VS'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=3286, version=35, score=0.2, payload={'preprocesse

In [43]:
reviews.points

[ScoredPoint(id=3362, version=36, score=0.5, payload={'preprocessed_data': 'Very poor sound quality ! Very poor sound quality !', 'parent_asin': 'B0047T79VS'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3099, version=33, score=0.33333334, payload={'preprocessed_data': 'Low quality sound Low quality sound! This is the worse quality you can get, is staticky. I used it for my laptop to play in my home theater and it sucks because it take away sound, base, quality & volumen. Do not buy if you are looking to use it for music', 'parent_asin': 'B0047T79VS'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=4548, version=48, score=0.25, payload={'preprocessed_data': "Bad quality sound. Don't waste your money.  Sound quality is bad.  It degrades the already compressed mp3 and sounds like AM radio.", 'parent_asin': 'B0047T79VS'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3286, version=35, score=0.2, payload={'preprocessed_data': "Very Low

### Define the reviews retrieval tool

In [45]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_prefiltered_reviews_data(query, parent_asins, k=5):

    query_embedding = get_embedding(query)

    qdrant_client = QdrantClient(url="http://localhost:6333")

    results = qdrant_client.query_points(
        collection_name="Amazon-reviews-collection-01",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_data"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
    }


def process_reviews_context(context):

    formatted_context = ""

    for id, chunk in zip(context["retrieved_context_ids"], context["retrieved_context"]):
        formatted_context += f"- ID: {id}, user review: {chunk}\n"

    return formatted_context


def get_formatted_reviews_context(query: str, parent_asins: list[str], top_k: int = 5) -> str:

    """Get the top k reviews matching a query for a list of prefiltered items.
    
    Args:
        query: The query to get the top k reviews for
        item_list: The list of item IDs to prefilter for before running the query
        top_k: The number of reviews to retrieve, this should be at least 20 if multipple items are prefiltered
    
    Returns:
        A string of the top k context chunks with IDs prepending each chunk, each representing a review for a given inventory item for a given query.
    """

    retrieved_context = retrieve_prefiltered_reviews_data(
        query,
        parent_asins,
        top_k
    )
    formatted_context = process_reviews_context(retrieved_context)

    return formatted_context

In [47]:
result = get_formatted_reviews_context("bad quality", ["B0047T79VS", "B0047T79VS"])
result

"- ID: B0047T79VS, user review: Very poor sound quality ! Very poor sound quality !\n- ID: B0047T79VS, user review: Low quality sound Low quality sound! This is the worse quality you can get, is staticky. I used it for my laptop to play in my home theater and it sucks because it take away sound, base, quality & volumen. Do not buy if you are looking to use it for music\n- ID: B0047T79VS, user review: Bad quality sound. Don't waste your money.  Sound quality is bad.  It degrades the already compressed mp3 and sounds like AM radio.\n- ID: B0047T79VS, user review: Very Low Quality Avoid this. Doesn't work. Good manufacturer, bad nonfunctional product. The sound quality is poor and this product's ability to use the Bluetooth protocol is limited.\n- ID: B0047T79VS, user review: Poor Sound Quality Not worth the money, and has....poor sound quality and is really degraded.  Not happy with the purchase, and will be looking for another product that is better quality!\n"